# Advanced Wind Turbine Simulation Workflow

* A more advanced turbine workflow will include many steps (Which have been showcased in previous examples)
* The process exemplified here includes:
  1. Weather data extraction from a MERRA dataset (windspeed, pressure, temperature)
  2. Spatial adjustment of the windspeeds
  3. Vertical projection of the wind speeds
  4. Wind speed density correction
  5. Power curve convolution
  6. Capacity Factor Estimation

In [ ]:
import reskit as rk

import numpy as np
import matplotlib.pyplot as plt

# Simulate a Single Location

In [ ]:
# Set some constants for later

TURBINE_CAPACITY = 4200  # kW
TURBINE_HUB_HEIGHT = 120  # meters
TURBINE_ROTOR_DIAMETER = 136  # meters
TURBINE_LOCATION = (6.0, 50.5)  # (lon, lat)

In [ ]:
# 1. Create a weather source, load, and extract weather variables
src = rk.weather.MerraSource(rk.TEST_DATA["merra-like"], bounds=[5, 49, 7, 52], verbose=False)

src.sload_elevated_wind_speed()
src.sload_surface_pressure()
src.sload_surface_air_temperature()

raw_windspeeds = src.get("elevated_wind_speed", locations=TURBINE_LOCATION, interpolation="bilinear")
raw_pressure = src.get("surface_pressure", locations=TURBINE_LOCATION, interpolation="bilinear")
raw_temperature = src.get("surface_air_temperature", locations=TURBINE_LOCATION, interpolation="bilinear")

print(raw_windspeeds.head())

In [ ]:
# 2. Vertically project wind speeds to hub height

roughness = rk.wind.roughness_from_clc(clc_path=rk.TEST_DATA["clc-aachen_clipped.tif"], loc=TURBINE_LOCATION)


projected_windspeed = rk.wind.apply_logarithmic_profile_projection(
    measured_wind_speed=raw_windspeeds,
    measured_height=50,  # The MERRA dataset offers windspeeds at 50m
    target_height=TURBINE_HUB_HEIGHT,
    roughness=roughness,
)

print(projected_windspeed.head())

In [ ]:
# 3. Apply density correction

pressure_corrected_windspeeds = rk.wind.apply_air_density_adjustment(
    wind_speed=projected_windspeed,
    pressure=raw_pressure,
    temperature=raw_temperature,
    height=TURBINE_HUB_HEIGHT,
)

pressure_corrected_windspeeds.head()

In [ ]:
# 4. Power curve estimation and convolution

power_curve = rk.wind.PowerCurve.from_capacity_and_rotor_diam(
    capacity=TURBINE_CAPACITY, rotor_diam=TURBINE_ROTOR_DIAMETER
)

convoluted_power_curve = power_curve.convolute_by_gaussian(scaling=0.06, base=0.1)

In [ ]:
# 5. Capacity factor estimation
capacity_factors = convoluted_power_curve.simulate(wind_speed=pressure_corrected_windspeeds)

capacity_factors

In [ ]:
plt.plot(capacity_factors)
plt.show()

# Simulate multiple locations at once (recommended)

In [ ]:
TURBINE_CAPACITY = 4200  # kW
TURBINE_HUB_HEIGHT = 120  # meters
TURBINE_ROTOR_DIAMETER = 136  # meters
TURBINE_LOCATION = np.array([(6.25, 51.0), (6.50, 51.0), (6.25, 50.75)])  # (lon,lat)


# 1
raw_windspeeds = src.get("elevated_wind_speed", locations=TURBINE_LOCATION, interpolation="bilinear")
raw_pressure = src.get("surface_pressure", locations=TURBINE_LOCATION, interpolation="bilinear")
raw_temperature = src.get("surface_air_temperature", locations=TURBINE_LOCATION, interpolation="bilinear")

# 2
roughness = rk.wind.roughness_from_clc(clc_path=rk.TEST_DATA["clc-aachen_clipped.tif"], loc=TURBINE_LOCATION)


projected_windspeed = rk.wind.apply_logarithmic_profile_projection(
    measured_wind_speed=raw_windspeeds,
    measured_height=50,  # The MERRA dataset offers windspeeds at 50m
    target_height=TURBINE_HUB_HEIGHT,
    roughness=roughness,
)

# 3
pressure_corrected_windspeeds = rk.wind.apply_air_density_adjustment(
    wind_speed=projected_windspeed,
    pressure=raw_pressure,
    temperature=raw_temperature,
    height=TURBINE_HUB_HEIGHT,
)

convoluted_power_curve = power_curve.convolute_by_gaussian(scaling=0.06, base=0.1)

# 4
power_curve = rk.wind.PowerCurve.from_capacity_and_rotor_diam(
    capacity=TURBINE_CAPACITY, rotor_diam=TURBINE_ROTOR_DIAMETER
)

# 5
capacity_factors = convoluted_power_curve.simulate(wind_speed=pressure_corrected_windspeeds)


# Print result
capacity_factors

In [ ]:
capacity_factors.plot()
plt.show()